# 🧠 Lab 1 — AI Risk Management: Audit Before You Deploy
### Cybersecurity Specialization • Colab Edition

> **Mission:** You are the AI Risk Analyst for a financial-services team. A prototype model predicts whether a person is likely to earn more than $50K and the business wants to use that signal to **accelerate eligibility review**.

Your job is **not** just to maximize accuracy. You must decide whether the system is safe enough to move forward.

### What you will do
1. Audit the dataset for quality and representation risks.
2. Train a baseline model and evaluate performance.
3. Measure outcome differences across **sex** and **race** groups.
4. Red-team the model with matched applicants.
5. Test a mitigation by removing sensitive features.
6. Add a **human-in-the-loop** decision zone.
7. Build a risk register and make a **GO / CONDITIONAL GO / NO-GO** recommendation.

**Important:** The Adult/Census Income dataset is used here as a teaching proxy. Income is **not** a valid stand-in for creditworthiness or an automatic basis for real financial decisions.


## 🎯 Scenario: The 4 PM Deployment Meeting

The product owner says:

> “The model is accurate enough. Can we deploy Friday?”

Security is worried about misuse. Legal is worried about discriminatory impact. Operations wants fewer manual reviews. The model team says the protected attributes improve prediction.

At the end of the lab, **you** must brief the team and defend your decision with evidence.


In [ ]:
# Install only what this lab needs
!pip install pandas scikit-learn matplotlib --quiet


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split

print("✅ Environment ready")


## 1. Load the Adult dataset

The loader looks for any `adult*.csv` file already uploaded to Colab. If it cannot find one, it asks you to upload the file.


In [ ]:
column_names = [
    "age", "workclass", "fnlwgt", "education", "education.num",
    "marital.status", "occupation", "relationship", "race", "sex",
    "capital.gain", "capital.loss", "hours.per.week", "native.country", "income"
]

matches = glob.glob("/content/adult*.csv")

if matches:
    file_path = matches[0]
else:
    from google.colab import files
    uploaded = files.upload()
    file_path = next(iter(uploaded))

raw = pd.read_csv(
    file_path,
    header=None,
    names=column_names,
    na_values="?",
    skipinitialspace=True
)

raw["income"] = raw["income"].astype(str).str.strip().str.replace(".", "", regex=False)

print(f"✅ Loaded: {os.path.basename(file_path)}")
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]}")
display(raw.head())


## 2. Data Risk Triage

Before training anything, look for evidence that could create downstream risk.

**Checkpoint:** If a dataset is historically biased, can a technically accurate model still reproduce that bias? What would you want to know before trusting this data?


In [ ]:
audit = pd.DataFrame({
    "missing_values": raw.isna().sum(),
    "missing_percent": (raw.isna().mean() * 100).round(2),
    "unique_values": raw.nunique()
}).sort_values("missing_percent", ascending=False)

print("Duplicate rows:", raw.duplicated().sum())
print("\nTarget distribution:")
display(raw["income"].value_counts(normalize=True).rename("proportion").to_frame())

print("\nColumns with the most missing data:")
display(audit.head(8))


In [ ]:
# Compare observed target rates across groups before modeling
def observed_rate_table(df, group_col):
    out = (
        df.assign(target=(df["income"] == ">50K").astype(int))
          .groupby(group_col)["target"]
          .agg(["count", "mean"])
          .rename(columns={"mean": "observed_>50K_rate"})
          .sort_values("count", ascending=False)
    )
    out["observed_>50K_rate"] = out["observed_>50K_rate"].round(3)
    return out

print("Observed outcome by sex")
display(observed_rate_table(raw, "sex"))

print("Observed outcome by race")
display(observed_rate_table(raw, "race"))


### 🔎 Investigator Question

Large group differences in the dataset do **not automatically prove model discrimination**. They are a warning that the model may learn historical or structural patterns.

Write down one plausible **data risk**, one **ethical risk**, and one **operational risk** you see so far.


## 3. Build the Baseline Model

The baseline intentionally includes `sex` and `race` so you can observe what happens when protected attributes are available to the model.


In [ ]:
model_data = raw[
    ["age", "education.num", "hours.per.week", "sex", "race", "income"]
].dropna().copy()

model_data["target"] = (model_data["income"] == ">50K").astype(int)

# Keep group labels separate for later audits
indices = np.arange(len(model_data))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.25,
    random_state=42,
    stratify=model_data["target"]
)

feature_cols = ["age", "education.num", "hours.per.week", "sex", "race"]
X_all = pd.get_dummies(model_data[feature_cols], drop_first=False)
y_all = model_data["target"].reset_index(drop=True)

X_train = X_all.iloc[train_idx]
X_test = X_all.iloc[test_idx]
y_train = y_all.iloc[train_idx]
y_test = y_all.iloc[test_idx]

baseline = RandomForestClassifier(
    n_estimators=250,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
baseline.fit(X_train, y_train)

pred = baseline.predict(X_test)
proba = baseline.predict_proba(X_test)[:, 1]

print("✅ Baseline trained")
print(classification_report(y_test, pred, digits=3))


In [ ]:
cm = confusion_matrix(y_test, pred)

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm)
ax.set_title("Baseline Confusion Matrix")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")

for (i, j), value in np.ndenumerate(cm):
    ax.text(j, i, value, ha="center", va="center")

plt.show()


## 4. Fairness Audit: Who Gets Different Outcomes?

Accuracy can hide subgroup harm. We will compare:

- **Selection rate** — fraction predicted `>50K`
- **True positive rate (TPR)** — fraction of actual positives correctly identified
- **False positive rate (FPR)** — fraction of actual negatives incorrectly labeled positive
- **Accuracy** — overall correctness within the group


In [ ]:
test_meta = model_data.iloc[test_idx][["sex", "race"]].reset_index(drop=True)
audit_results = test_meta.copy()
audit_results["actual"] = y_test.reset_index(drop=True)
audit_results["predicted"] = pred
audit_results["probability"] = proba

def group_metrics(df, group_col):
    rows = []
    for group, g in df.groupby(group_col):
        tn, fp, fn, tp = confusion_matrix(
            g["actual"], g["predicted"], labels=[0, 1]
        ).ravel()

        rows.append({
            group_col: group,
            "n": len(g),
            "selection_rate": g["predicted"].mean(),
            "TPR": tp / (tp + fn) if (tp + fn) else np.nan,
            "FPR": fp / (fp + tn) if (fp + tn) else np.nan,
            "accuracy": (g["actual"] == g["predicted"]).mean()
        })

    return pd.DataFrame(rows).sort_values("n", ascending=False)

sex_metrics = group_metrics(audit_results, "sex")
race_metrics = group_metrics(audit_results, "race")

print("Fairness metrics by sex")
display(sex_metrics.round(3))

print("Fairness metrics by race")
display(race_metrics.round(3))


In [ ]:
# Visualize selection-rate differences
for group_col, table in [("sex", sex_metrics), ("race", race_metrics)]:
    plot_table = table.sort_values("selection_rate")
    plt.figure(figsize=(7, 4))
    plt.barh(plot_table[group_col], plot_table["selection_rate"])
    plt.xlabel("Predicted >50K selection rate")
    plt.title(f"Selection Rate by {group_col.title()}")
    plt.xlim(0, max(0.5, plot_table["selection_rate"].max() + 0.05))
    plt.show()


### 🚨 Decision Point 1

Suppose overall model accuracy looks strong, but selection rates or error rates differ sharply across groups.

Would you:
- deploy because aggregate accuracy is acceptable,
- stop deployment,
- or allow only a controlled pilot with additional safeguards?

Record the evidence you would cite.


## 5. Explainability Check: What Is the Model Using?

Feature importance does **not** prove causation, but it helps identify where to investigate.


In [ ]:
importance = (
    pd.Series(baseline.feature_importances_, index=X_train.columns)
      .sort_values(ascending=False)
      .head(12)
)

display(importance.rename("importance").to_frame())

plt.figure(figsize=(8, 5))
plt.barh(importance.index[::-1], importance.values[::-1])
plt.title("Top Baseline Feature Importances")
plt.xlabel("Random Forest feature importance")
plt.show()


## 6. Red-Team Test: Matched Applicants

Now test a simple but powerful question:

> If two records are identical except for a protected attribute, can the model's prediction change?

This is not a complete fairness test, but it can expose **direct sensitivity** to protected features.


In [ ]:
def make_candidate(age=40, education_num=13, hours=40, sex="Male", race="White"):
    row = pd.DataFrame([{
        "age": age,
        "education.num": education_num,
        "hours.per.week": hours,
        "sex": sex,
        "race": race
    }])
    encoded = pd.get_dummies(row, drop_first=False)
    return encoded.reindex(columns=X_train.columns, fill_value=0)

profiles = []
for sex in sorted(model_data["sex"].dropna().unique()):
    for race in sorted(model_data["race"].dropna().unique()):
        candidate = make_candidate(sex=sex, race=race)
        p = baseline.predict_proba(candidate)[0, 1]
        profiles.append({"sex": sex, "race": race, "predicted_probability": p})

matched_results = pd.DataFrame(profiles).sort_values(
    "predicted_probability", ascending=False
)

display(matched_results.round(3))

spread = (
    matched_results["predicted_probability"].max()
    - matched_results["predicted_probability"].min()
)

print(f"Probability spread for otherwise matched records: {spread:.3f}")


### 🔥 Red-Team Challenge

Try changing the matched applicant's `age`, `education_num`, or `hours` in `make_candidate()`.

Can you find a profile where changing only `sex` or `race` crosses the model's 0.50 decision threshold? If you can, explain why that would matter in a real deployment.


## 7. Mitigation Experiment: Remove Protected Attributes

A common first step is to remove protected attributes from the model. This may reduce direct dependence, but **it does not guarantee fairness** because other variables may still act as proxies.


In [ ]:
reduced_features = ["age", "education.num", "hours.per.week"]

X_reduced = model_data[reduced_features].reset_index(drop=True)
Xr_train = X_reduced.iloc[train_idx]
Xr_test = X_reduced.iloc[test_idx]

mitigated = RandomForestClassifier(
    n_estimators=250,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
mitigated.fit(Xr_train, y_train)

pred_m = mitigated.predict(Xr_test)
proba_m = mitigated.predict_proba(Xr_test)[:, 1]

mitigated_results = test_meta.copy()
mitigated_results["actual"] = y_test.reset_index(drop=True)
mitigated_results["predicted"] = pred_m
mitigated_results["probability"] = proba_m

comparison = pd.DataFrame({
    "model": ["Baseline", "Protected attributes removed"],
    "accuracy": [
        accuracy_score(y_test, pred),
        accuracy_score(y_test, pred_m)
    ],
    "precision": [
        precision_score(y_test, pred),
        precision_score(y_test, pred_m)
    ],
    "recall": [
        recall_score(y_test, pred),
        recall_score(y_test, pred_m)
    ]
})

display(comparison.round(3))


In [ ]:
baseline_sex = group_metrics(audit_results, "sex").set_index("sex")
mitigated_sex = group_metrics(mitigated_results, "sex").set_index("sex")

fairness_compare = pd.DataFrame({
    "baseline_selection_rate": baseline_sex["selection_rate"],
    "mitigated_selection_rate": mitigated_sex["selection_rate"],
    "baseline_TPR": baseline_sex["TPR"],
    "mitigated_TPR": mitigated_sex["TPR"]
})

display(fairness_compare.round(3))

def gap(series):
    return float(series.max() - series.min())

print(
    "Baseline sex selection-rate gap:",
    round(gap(baseline_sex["selection_rate"]), 3)
)
print(
    "Mitigated sex selection-rate gap:",
    round(gap(mitigated_sex["selection_rate"]), 3)
)


### 🧪 Investigator Question

Did removing protected attributes:
1. materially reduce group gaps,
2. preserve useful model performance,
3. solve the fairness problem completely?

If the answer to #3 is “no,” what other controls might be needed?


## 8. Human-in-the-Loop Control

Instead of forcing the model to make every decision, create an **uncertainty zone**:

- probability **≥ 0.70** → easy positive
- probability **≤ 0.30** → easy negative
- probability **between 0.30 and 0.70** → **escalate to a human reviewer**

This turns human review into a risk control rather than an afterthought.


In [ ]:
def hitl_decision(prob, low=0.30, high=0.70):
    if prob >= high:
        return "AUTO-POSITIVE"
    if prob <= low:
        return "AUTO-NEGATIVE"
    return "HUMAN REVIEW"

hitl = mitigated_results.copy()
hitl["decision"] = hitl["probability"].apply(hitl_decision)

print("Overall decision routing")
display(hitl["decision"].value_counts(normalize=True).rename("share").to_frame().round(3))

print("Human-review rate by sex")
review_by_sex = (
    hitl.assign(human_review=hitl["decision"].eq("HUMAN REVIEW"))
        .groupby("sex")["human_review"]
        .agg(["count", "mean"])
        .rename(columns={"mean": "human_review_rate"})
)
display(review_by_sex.round(3))


### ⚖️ Decision Point 2

Human review sounds safer, but it creates new risks:

- reviewer inconsistency,
- automation bias (“the AI is probably right”),
- slower service,
- unequal escalation rates across groups,
- inadequate audit logging.

**Challenge:** Change the thresholds to `0.20 / 0.80` or `0.40 / 0.60`. What happens to the percentage of cases sent to humans?


## 9. Build a Risk Register

Use **Likelihood × Impact** to prioritize risks. Scores are intentionally editable — the point is to defend your reasoning.


In [ ]:
risk_register = pd.DataFrame([
    ["Historical / representation bias", "Data", 4, 5, "Subgroup outcome differences in source data", "Dataset review + representative validation data"],
    ["Disparate model outcomes", "Ethical", 4, 5, "Selection/error-rate gaps across groups", "Fairness testing + thresholds + governance review"],
    ["Direct use of protected attributes", "Governance", 4, 5, "Baseline model includes sex/race", "Remove or strictly justify protected-feature use"],
    ["Low explainability", "Technical", 3, 4, "Random forest is not inherently transparent", "Explanation tooling + model cards + review"],
    ["Automation bias", "Operational", 3, 4, "Humans may over-trust model recommendations", "Reviewer training + independent judgment prompts"],
    ["Model/data drift", "Operational", 3, 4, "Population and labor patterns can change", "Monitoring + drift thresholds + retraining plan"],
])

risk_register.columns = [
    "risk", "category", "likelihood", "impact",
    "evidence", "possible_control"
]
risk_register["score"] = risk_register["likelihood"] * risk_register["impact"]
risk_register["priority"] = pd.cut(
    risk_register["score"],
    bins=[0, 6, 12, 19, 25],
    labels=["LOW", "MEDIUM", "HIGH", "CRITICAL"]
)

display(risk_register.sort_values("score", ascending=False))


## 10. Map the Work to the NIST AI RMF

| NIST AI RMF Function | What you did in this lab |
|---|---|
| **GOVERN** | Defined risk ownership, acceptable use, and human review controls |
| **MAP** | Identified context, affected groups, intended use, and misuse risk |
| **MEASURE** | Measured accuracy, subgroup metrics, matched-case sensitivity, and risk scores |
| **MANAGE** | Tested mitigation, added escalation controls, and prioritized residual risks |

The important idea is that AI risk management is a **continuous lifecycle**, not a one-time accuracy test.


# 🧑‍⚖️ Final Challenge — The Deployment Review Board

You have five minutes before the 4 PM meeting.

Make one recommendation:

### 🟢 GO
Risk is within tolerance and controls are sufficient.

### 🟡 CONDITIONAL GO
Allow a limited pilot only if specific controls are implemented first.

### 🔴 NO-GO
Do not deploy until major risks are addressed.

Your briefing must include:

1. **Two pieces of quantitative evidence**
2. **The highest-priority risk**
3. **One mitigation that helped**
4. **One residual risk that remains**
5. **The role of human review**
6. **One monitoring metric you would track after deployment**

> **Stretch goal:** Argue the opposite recommendation from the one you chose. What evidence would the opposing team use?


## Optional Instructor Challenge: Trigger a Drift Event

Pretend the operating environment changes and the average weekly work pattern shifts. Re-run the model on a modified copy of the test set and compare the positive prediction rate.

```python
drifted = Xr_test.copy()
drifted["hours.per.week"] = (drifted["hours.per.week"] - 8).clip(lower=1)

original_rate = mitigated.predict(Xr_test).mean()
drifted_rate = mitigated.predict(drifted).mean()

print("Original positive rate:", round(original_rate, 3))
print("Drifted positive rate: ", round(drifted_rate, 3))
print("Change:", round(drifted_rate - original_rate, 3))
```

**Question:** At what size of change should monitoring trigger investigation or rollback?
